In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

In [3]:
from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    _fetch_mediawiki_file_metadata,
    _source_from_url,
    form_document_record,
    )

2026-05-30 20:20:07,675 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-05-30 20:20:07,844 [INFO] Translation is enabled. Using GCP translator
2026-05-30 20:20:07,845 [INFO] Using Google Cloud translation API
2026-05-30 20:20:07,845 [INFO] GoogleCloudTranslator using REST API
2026-05-30 20:20:08,198 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [4]:
db = Database()

2026-05-30 20:20:08,403 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.95    39.00       0.00           24


In [ ]:
doc_url="https://upload.wikimedia.org/wikipedia/commons/archive/d/d7/20230312214857!R5069-32-0002.pdf"

In [ ]:
_source_from_url(doc_url)

In [ ]:
form_document_record(doc_url)

In [ ]:
t= form_document_record(doc_url)['title']
mr=_fetch_mediawiki_file_metadata([t],source="commons")
mr.get(t)

In [ ]:
runtime = Runtime()

In [ ]:
def do_pass(limit=20, source="commons"):
    rec, _ = db.scan("Documents", view_name="BD:Missing Metadata", fields="url", limit=limit)
    doc_recs = [form_document_record(r["url"]) for r in rec]
    return doc_recs
    subset = [rec for rec in doc_recs if _source_from_url(rec["url"]) == source]
    subset_titles = [rec["title"] for rec in subset]
    #subset_titles.extend(["|", "[", "]", ""])
    #subset_titles.append("MISSING_TITLE")
    #return subset_titles
    metadata_recs = _fetch_mediawiki_file_metadata(subset_titles, source)
    return metadata_recs

In [ ]:
def do_pass_2(limit=10):
    rec, _ = db.scan("Documents", view_name="BD:Needs Metadata", fields="url", limit=limit)
    urls = [r["url"] for r in rec]
    if urls:
        print(f"updating document metadata len=={len(urls)}")
        runtime.update_documents_to_database(urls)

In [ ]:
do_pass_2(limit=100)

In [ ]:
runtime.run_database_housekeeping()

In [ ]:
runtime._database_updater.update_doc_records([doc_url])

In [5]:
def update_wiki_flag(db):
    limit = 1000
    while True:
        recs, _ = db.scan("Documents", limit=limit, view_name="Wiki Flag Not Set", fields="url")
        recs = [r for r in recs if "wikimedia" in r['url'] or "wikisource" in r['url']]
        for r in recs:
            r['wiki'] = True
        if not recs:
            break
        print(len(recs))
        db.write("Documents", recs)            

In [6]:
update_wiki_flag(db)

1000
578
